In [1]:
# Loaded data

import pandas as pd

df = pd.read_parquet("../data/processed/online_retail_after_ingestion.parquet")

**Handling Cancellations & Returns**

Objective

Correctly identify and classify non-standart transactions (cancellations and returns) so that:
- revenue metrics are accurate
- return rates can be analyzed
- customer and product KPIs are not distorted

This step directly impacts:
- total revenue
- average order value
- customer ranking
- product performance

Business Context:
In the Online Retail II dataset:
- Cancellations are represented by invoices started with "C"
- Returns are represented by negative quantity

These two cases are not the same and must be handled differently

In [ ]:
# Identify Cancellations

# Business meaning:
# - Cancellations usually represent orders that never completed
# - Incluing them in revenue would overstate rates

df["is_cancellation"] = df["Invoice"].astype(str).str.startswith("C")
# At this stage we only flag, not return

# Identify Returns (Negative Quantities)
df["is_return"] = df["Quantity"] < 0

# Business meaning:

# Returns represent completed sales that were later reversed
# They are essential for:
# - return-rate analysis
# - product quality signals
# - loss estimation

# Returns should now be silently dropped

# Combined Overview
df[['is_cancellation','is_return']].mean()

# This shows the share of trasnactions afected
# Helps assess materiality before exclusions

# Revenue Impact Preview (Before Cleaning)

# created temporary field for diagnostic purposes
df['raw_revenue'] = df['Quantity'] * df['Price']

# Now inspect revenue contribution
df.groupby(['is_cancellation','is_return'])['raw_revenue'].sum()

# Key question asnwered here:
# - How much revenue comes from cancelled invoices
# - How much revenue is reversed by returns

# This informs what we remove what we keep

# Cleaning decision

# Cancellation
# - Exclude from revenue calculation
# - Exclude from order counts
# - Exclude from customer purchase history

# Returns
# - Keep in dataset
# - Keep negative quantities
# Use for:
# - return-rate metrics
# - keep negative quantities
# - net revenue
# - product quality analysis

# Apply Cancellation Exclusion

df_clean = df.loc[~df['is_cancellation']].copy()

# Validate Resulting Dataset
{
    "rows_before": len(df),
    "rows_after": len(df_clean),
    "cancellations_removed": (df['is_cancellation']).sum()
}

# Recompute Revenue

df_clean["revenue"] = df_clean["Quantity"] * df_clean["Price"]

# Result
# - Positive revenue - sales
# - Negative revenue - returns

# It enables
# - net revenue analysis
# - return impact quntification

# Sanity Check
df_clean["revenue"].describe()

# Look for:
# - reasonable max value
# - presence of negative values (expected)
# - no extreme distortions

# Summary of This Step:
# - fully removed cancellations
# - returns are explecetly preserved
# - revenue logic is business consistent
# - dataset is safe for customer and product analysis

# 2.3 Missing & Invalid Customers


**Missing & Invalid Customers**

**Objective**

Decide how to handle transactions without a valid CustomerID in a way that:

- preserves accurate revenue reporting
- avoids corrupting customer-level analytics
- keeps decisions transparent and justifiable

**Business Context**

In the Online Retail II dataset:

- CustomerID is required to identify unique customers
- Some transactions do not have a CustomerID
- These rows may still represent valid sales

Key principle:

**Transactions without a customer can contribute to revenue,
but cannot contribute to customer behavior analysis.**

In [3]:
# Identify Missing CustomerID
df_clean.head()
df_clean['has_customer'] = df_clean['Customer ID'].notna()


# Impact Assessment
# Share of transactions without customer id
df_clean['has_customer'].value_counts(normalize=True)

# Revenue Impact
df_clean.groupby("has_customer")['revenue'].sum()

# Customer-Level Risk Check
# df_clean.loc[~df_clean["has_customer"], "Invoice"].nunique()
df_clean[~df_clean["has_customer"]]["Invoice"].nunique()

# Cleaning Decision (Explicit) 

# Customer-level analysis
# - Exclude rows without CustomerID
 
# Revenue & product analysis

# - Keep rows without CustomerID

# Create Customer Valid Dataset
df_customers = df_clean.loc[df_clean["has_customer"]].copy()

# df_clean: revenue-complete dataset
# df_customers: customer-safe dataset

# Validation Checks
{
    "total_rows": len(df_clean),
    "customer_rows": len(df_customers),
    "rows_excluded_from_customer_analysis": len(df_clean) - len(df_customers)
}

# Revenue Validation
df_customers['revenue'].sum() / df_clean['revenue'].sum()

# This tells you:
# - what % of revenue is analyzable at customer level

# Summary of This Step

# After this step:
# - missing customers are explicitly flagged
# - customer-level dataset is clean and safe
# - revenue totals remain complete and honest
# - trade-offs are documented and measurable

df_clean.to_parquet("../data/processed/online_retail_after_cleaning.parquet")

# save cleaned dataframe to share among different ipynb files